# NIST line list

Goal: test `linelist.query_nist`, which queries the
NIST Atomic Spectra Database, and compare it with the offline Verner+96
table (`linelist.get_lines`).

Only resonance lines are kept: electric-dipole transitions from the ground
level ($E_i = 0$) with both $f$ and $A_{ki}$ available. The damping constant
$\Gamma$ is the total decay rate of the upper level, i.e. the sum of $A_{ki}$
over every transition NIST lists from that level (to any lower level).
Wavelengths in Angstrom, vacuum.

In [1]:
import numpy as np
from astropy.table import Table

from spectral_diag.linelist import LineSelection, get_lines, query_nist

## Query

`f_min` drops the intercombination / very weak lines (e.g. SiII 2329, 2335).

In [2]:
species = ["CIV", "SiII", "FeII"]
selection = LineSelection(wave_min=1150, wave_max=2700, f_min=1e-3)
nist = {sp: query_nist(sp, selection)
        for sp in species}
for sp, lines in nist.items():
    print(f"{sp}: {len(lines)} lines")

CIV: 2 lines
SiII: 6 lines
FeII: 10 lines


## Comparison with the offline table

Lines are matched by rounded wavelength. Relative differences in $f$ and
$\Gamma$ are (NIST - table) / table.

In [3]:
rows = []
for sp in species:
    table = {round(line.wave_rest): line for line in get_lines(sp)}
    for n in nist[sp]:
        t = table.get(round(n.wave_rest))
        rows.append((
            n.name, n.wave_rest, n.f, n.gamma,
            np.nan if t is None else n.wave_rest - t.wave_rest,
            np.nan if t is None else n.f / t.f - 1,
            np.nan if t is None else n.gamma / t.gamma - 1,
        ))
comparison = Table(rows=rows, names=["name", "wave_rest", "f", "gamma",
                                     "d_wave", "rel_d_f", "rel_d_gamma"])
for col, fmt in [("wave_rest", ".4f"), ("f", ".3e"), ("gamma", ".3e"),
                 ("d_wave", ".4f"), ("rel_d_f", "+.3f"), ("rel_d_gamma", "+.3f")]:
    comparison[col].info.format = fmt
comparison

name,wave_rest,f,gamma,d_wave,rel_d_f,rel_d_gamma
str13,float64,float64,float64,float64,float64,float64
CIV 1548.204,1548.2043,1.900e-01,2.650e+08,0.0093,+0.000,+0.004
CIV 1550.784,1550.7840,9.520e-02,2.640e+08,0.0140,+0.000,+0.000
SiII 1190.416,1190.4158,2.770e-01,4.103e+09,-0.0002,-0.055,+0.001
SiII 1193.290,1193.2897,5.750e-01,4.090e+09,-0.0003,-0.015,-0.002
SiII 1260.422,1260.4221,1.220e+00,3.043e+09,0.0001,+0.034,+0.024
SiII 1304.370,1304.3702,9.280e-02,9.870e+08,0.0002,+0.065,-0.029
SiII 1526.707,1526.7066,1.330e-01,1.133e+09,-0.0004,+0.008,+0.011
SiII 1808.013,1808.0130,2.490e-03,2.863e+06,0.0000,-0.027,-0.084
FeII 1260.536,1260.5355,2.000e-02,1.236e+09,nan,+nan,+nan


Lines missing from the table (NaN differences) are NIST resonance lines
not included in our hand-written selection. Some table lines can be
missing from NIST if it lacks $f$ or $A_{ki}$ for them (e.g. FeII 2367).

FeII oscillator strengths differ by up to ~30 % (1608, 2374, 2586): NIST
uses more recent measurements than Verner+96, closer to Morton 2003.
Since N scales as 1/f in the linear regime, the choice of source matters.